In [ ]:
import pandas as pd
import struct
from sqlalchemy import create_engine, event
from azure.identity import InteractiveBrowserCredential
import pyodbc

#  server and database 
SERVER   = ""
DATABASE = ""

#  This opens a browser window to log in with your Microsoft account
# I got this from error code 0x80070057 and went to microsoft help forums
credential = InteractiveBrowserCredential()
token = credential.get_token("get token")

#  Build the token in the format pyodbc expects
# Got this from help forums
token_bytes = token.token.encode("find utf")
token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)

# Connection string if you can get the connection format in settings for azure database on website
conn_str = (
    # Find in connection settings for database
    f"driver"
    f"server"
    f"database"
    f"encrypt"
    f"trustservercertificate"
)

# Create engine and inject the token 
SQL_COPT_SS_ACCESS_TOKEN = "find token "

def connect():
    conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})
    return conn

engine = create_engine("mssql+pyodbc://", creator=connect)

#  Load CSV and push to Azure SQL 
df = pd.read_csv("path/file.csv")

df.to_sql(
    name="breweries",
    con=engine,
    schema="dbo",
    if_exists="append",
    index=False
)

print(f"✅ Loaded {len(df)} rows into dbo.breweries")